# Verification of the modulo-125 identities in Sage

Following the other playgrounds, we define the relations, load the precomputed source modules and recorded intermediate elements, and check their defining equations.

Working precision is **625**; the classification modulus is **125**. The complete signed Manin sources, including torsion, are used. At recursive degrees we check transfer compatibility and generation by inherited elements together with the recorded complement.

The Manin presentations and Hecke matrices are inputs. This notebook checks finite-source conditions; all-weight propagation, passage to eigenvalues and strong realization are the separate arguments in the manuscript. No search for division outputs is performed.


In [9]:
import gzip
import hashlib
import json
import sys
from pathlib import Path
from functools import lru_cache
import numpy as np
from sage.all import *

ROOT = Path.cwd()
if not (ROOT / 'python').is_dir():
    raise RuntimeError('Open this notebook from the repository directory.')
sys.path.insert(0, str(ROOT / 'python'))
from verify_hecke_relations import relation_spec
from p5_mod125 import load_mod125_parameters

p = 5
relation_period = 100
VERIFICATION_DIRECTORY = ROOT / 'verification_data/mod125_compact'
SUPPLEMENTARY_SOURCE_DIRECTORY = ROOT / 'source_data/p5_mod625_lower_minus'
SUPPLEMENTARY_VERIFICATION_DIRECTORY = ROOT / 'verification_data/mod125_lower_minus/packets'


## 1. Define all the relations

Put $r=(d+50q)\bmod100$ and $t_n=n^{125q}T_n$ on the signed source with sign $(-1)^q$. The twist is applied exactly once.

The ordinary relation is
$$
t_2^2-h_{r,0}-h_{r,1}t_{19}-h_{r,2}t_{19}^2\equiv0\pmod{125}.
$$
The two division relations are
$$
\mathcal Z_{19,q}=\mathfrak D_{t_{19},5},\qquad
\mathcal B_{r,q}=\mathfrak D_{1,25}\circ N_r(\mathcal Z_{19,q}),
$$
where $H_r(X)=\alpha_r+\beta_rX+\gamma_rX^2$ and
$N_r(X)=(X^5-X)^2-5H_r(X)(X^5-X)$. Both must have full domain.

Finally put $E_\ell(X)=(1-(X-\ell)^4)^5$,
$\mathcal E_{\ell,q}=E_\ell(\mathcal Z_{19,q})$, and
$\mathcal J_{r,\ell,q}=\mathcal E_{\ell,q}\circ\mathcal B_{r,q}\circ\mathcal E_{\ell,q}$.
For each $\ell=0,\ldots,4$, test
$$
P_{r,\ell}(\mathcal J_{r,\ell,q})\circ\mathcal E_{\ell,q}
\equiv0\pmod5.
$$

The parameter rows below come from the repository's table. The terminal polynomials are powered over $\mathbf F_5$ **before** lifting coefficients. All relation definitions are grouped here, outside the replay functions. The archived name V denotes $\mathcal Z_{19,q}$; B, E0, J0, etc. retain their archive names solely to identify the saved intermediate elements.


In [10]:
m = 4
R = Integers(p^m)
S.<X> = PolynomialRing(ZZ)
K.<Y> = PolynomialRing(GF(5))
parameters = load_mod125_parameters()
H_r, N_r, P_rl = {}, {}, {}
E_l = {ell:(1-(X-ell)^4)^5 for ell in range(5)}
for r,row in parameters.items():
    H_r[r] = row['alpha']+row['beta']*X+row['gamma']*X^2
    N_r[r] = (X^5-X)^2-5*H_r[r]*(X^5-X)
    v_r = row['v0']
    u_r = ((v_r^5-v_r)//5) % 5
    lam = v_r % 5
    Lambda_r = {lam,(lam+2)%5,(lam-2)%5}
    assert H_r[r](lam) % 5 == 2*u_r % 5
    for ell in range(5):
        root_polynomial = (K.one() if ell not in Lambda_r else
                           Y+u_r^2 if ell == lam else
                           prod(Y-t^2+GF(5)(H_r[r](ell))*t for t in range(5)))
        P_rl[r,ell] = S([ZZ(c) for c in (root_polynomial^4).list()])

def make_specification(r):
    """
    Encode the ordinary identity, two full-domain conditions and five selectors.
    Keep polynomial presentations and ordered compositions distinct.
    In particular, neither copy of E_l in E_l B E_l is removed on the source.
    Return the complete presentation in the recorded-data format.
    """
    row = parameters[r]
    names = ['T2','T19','V','B']
    for ell in range(5):
        names.extend((f'E{ell}',f'J{ell}'))
    ring = PolynomialRing(R,names=names)
    variables = dict(zip(names,ring.gens()))
    T2,T19,V,B = (variables[name] for name in names[:4])
    def terms(F):
        """Encode the expanded polynomial using integer coefficient lifts."""
        return [[str(ZZ(c)),[int(e) for e in powers]]
                for powers,c in ring(F).dict().items()]
    relations = [
        {'name':'ordinary_joint','polynomial':terms(
            T2^2-row['h0']-row['h1']*T19-row['h2']*T19^2),'terminal_power':int(3)},
        {'name':'V_full_domain','polynomial':terms(V),'terminal_power':int(0)},
        {'name':'B_full_domain','polynomial':terms(B),'terminal_power':int(0)},
    ]
    presentations = []
    for ell in range(5):
        E,J = f'E{ell}',f'J{ell}'
        presentations.extend((
            {'variable':E,'polynomial':terms(E_l[ell](V))},
            {'variable':J,'factors':[E,'B',E]},
        ))
        relations.append({'name':f'selector_{ell}',
                          'polynomial':terms(P_rl[r,ell](variables[J])),
                          'input_presentation':E,'terminal_power':int(1)})
    return {
        'schema':'hecke.relations.v1','coefficient_modulus':int(625),
        'variables':names,'hecke_operators':{'T2':int(2),'T19':int(19)},
        'divisions':[
            {'variable':'V','numerator':terms(T19),'power':int(1)},
            {'variable':'B','numerator':terms(N_r[r](V)),'power':int(2)},
        ],
        'presentations':presentations,'relations':relations,
        'witness_semantics':'independent_monomials','max_howell_dimension':int(4096),
    }

specifications = {r:make_specification(r) for r in range(0,100,2)}
plan = json.loads((ROOT/'relations/p5_mod125_native.json').read_text())
assert {str(r):spec for r,spec in specifications.items()} == plan['relation_specifications']
groups = {'mod125':{
    'exponent':m,'ring':R,'degrees':tuple(range(0,3250,2)),
    'threshold':750,'specifications':specifications,
    'sources':ROOT/'source_data/p5_mod625_recursive',
}}
assert 2500 % 100 == 0 and (-750+50) % 100 == 0
assert power_mod(2,125*4,625) == power_mod(19,125*4,625) == 1
print('Working modulus:',625,'| classification modulus:',125)
print('Lower degrees: 0,...,748, even, q=0')
print('Induction base: 750,...,3248, even, q=0,1,2,3')
print('Requested cases:',375+1250*4)


Working modulus: 625 | classification modulus: 125
Lower degrees: 0,...,748, even, q=0
Induction base: 750,...,3248, even, q=0,1,2,3
Requested cases: 5375


## 2. Recover intermediate elements and check the equations

For $5^a y=u$, the recorded data start with a coordinatewise preimage. Some files add a specified element of the kernel of multiplication by $5^a$; others provide reusable matrices of division outputs. Every defining equation is checked, including the final membership in $5^bM$.

Polynomial presentations and ordered products are expanded without treating the divided relations as single-valued operators. A recorded common chain is a permissible choice of intermediate elements; it does not impose additional identities on the source.

At recursive degrees we first verify the lower sources, then the transfer homomorphisms, Hecke compatibility, and generation by inherited images plus the complement. Supplementary lower-minus files are used when needed. The archive identifier checks file consistency, not mathematical correctness.

The successful-case cache lasts only for this notebook session. Rerun this helper cell after changing the definitions or source files.


In [11]:
GENERATOR_BATCH_SIZE = 4

def compact_json(value):
    """
    Serialize the source data and presentation in the archive's JSON format.
    Return the text used to identify the corresponding verification data.
    Values must already be JSON-compatible, including Python integers.
    """
    return json.dumps(value, separators=(',', ':'), ensure_ascii=False)

def nim_sequence(values):
    """
    Encode a sequence of integers in the archive's '@[1, 2, ...]' format.
    This is used only for the identifier of the recorded data; no Nim
    computation is performed.
    """
    return '@[' + ', '.join(str(int(v)) for v in values) + ']'

def mixed_zero(value, moduli):
    """
    Check equality to zero in the source module in cyclic coordinates.
    
    The rows represent elements of M = direct_sum_j Z/moduli[j]Z.
    Each column is reduced modulo its own cyclic order, rather than the
    ambient modulus. Return True precisely when all rows represent zero.
    For the zero module, this condition holds vacuously.
    """
    return all(ZZ(value[i,j]) % order == 0 for i in range(value.nrows())
               for j, order in enumerate(moduli))

def canonical_preimage(rhs, divisor, moduli):
    """
    Recover the recorded y in the relation rhs = divisor*y on M.
    
    The rows of rhs are elements of M in cyclic coordinates. The divisor
    and cyclic orders are powers of the same prime. In each coordinate,
    take the least nonnegative residue of rhs and divide it by divisor,
    using the coordinatewise choice specified by the recorded data.
    
    Return the rows y after checking divisor*y = rhs in M. Failure of
    the required divisibility raises ArithmeticError. These are choices
    of intermediate elements, not a globally defined divided endomorphism.
    """
    output = zero_matrix(rhs.base_ring(), rhs.nrows(), rhs.ncols())
    for i in range(rhs.nrows()):
        for j, order in enumerate(moduli):
            entry = ZZ(rhs[i,j]) % order
            if entry % gcd(divisor, order):
                raise ArithmeticError(f'Division equation fails in coordinate ({i},{j})')
            output[i,j] = entry // divisor
    assert mixed_zero(divisor*output-rhs, moduli)
    return output


def polynomial_matrix(terms, variables, matrices, ring, rank):
    """
    Evaluate an ordinary polynomial in the supplied matrices.
    Apply the rightmost variable first, using the row-vector convention.
    A missing variable with positive exponent is rejected.
    """
    result = zero_matrix(ring, rank, rank)
    for coefficient,powers in terms:
        term = identity_matrix(ring,rank)
        for name,power in reversed(list(zip(variables,powers))):
            if power:
                term *= matrices[name]^power
        result += ZZ(coefficient)*term
    return result

@lru_cache(maxsize=int(12))
def load_source(group_name, d, q):
    """
    Load the signed source, its ordinary Hecke actions and transfer maps.
    Apply the orientation exactly once. Check that each action respects
    the cyclic orders. The presentation and Hecke descent are trusted inputs.
    Only a small number of sources are retained in memory.
    """
    group = groups[group_name]
    ring, m = group['ring'], group['exponent']
    path = group['sources'] / f'degree_{d}.npz'
    prefix = f'q{q%2}_'
    if q % 2 and d < group['threshold']:
        path = SUPPLEMENTARY_SOURCE_DIRECTORY / f'degree_{d}.npz'
    with np.load(path, allow_pickle=False) as archive:
        metadata = json.loads(bytes(archive['metadata_json']).decode())
        assert metadata['prime'] == 5 and metadata['exponent'] == m and metadata['degree'] == d
        orders = [int(5^int(e)) for e in archive[prefix+'order_exponents']]
        assert all(1 < order <= 5^m and (5^m) % order == 0 for order in orders)
        rank = len(orders)
        matrices = {}
        for name,index in [('T2',2),('T19',19)]:
            raw = archive[prefix+name]
            assert raw.shape == (rank,rank)
            action = matrix(ring,rank,rank,[int(x) for x in raw.flat])
            action *= ring(power_mod(index,q*5^(m-1),5^m))
            assert all(orders[i]*ZZ(action[i,j]) % orders[j] == 0
                       for i in range(rank) for j in range(rank))
            matrices[name] = action
        maps = {}
        recursive = bool(archive[prefix+'recursive'][0])
        if recursive:
            for name in ('transfer_a','transfer_b','complement_images'):
                raw = archive[prefix+name]
                maps[name] = matrix(ring,raw.shape[0],raw.shape[1],
                                    [int(x) for x in raw.flat])
    return orders, matrices, maps, recursive

def select_complement(inherited, complement, rank, ring):
    """
    Select complement rows completing the transferred rows to generators.
    Perform elimination over F_5 in archive order. Full rank proves generation
    of the finite source by Nakayama's lemma. Return the selected original rows.
    """
    pivots = {}
    selected = []
    for block,rows in enumerate((inherited,complement)):
        for index,row in enumerate(rows.rows()):
            v = vector(GF(5),row)
            for j in range(rank):
                if not v[j]:
                    continue
                if j in pivots:
                    v -= v[j]*pivots[j]
                else:
                    v /= v[j]
                    pivots[j] = v
                    if block == 1:
                        selected.append(index)
                    break
            if len(pivots) == rank:
                return complement.matrix_from_rows(selected)
    if rank == 0:
        return zero_matrix(ring,0,0)
    raise ArithmeticError('Transferred rows and complement do not generate the source.')

def packet_binding(spec, metadata, orders, operators, m, inputs):
    """
    Identify the recorded intermediate elements for these exact inputs.
    Include the tested complement rows as well as the source and relations.
    The identifier prevents mismatched files; replay checks the equations.
    """
    parts = [compact_json(spec), f'5:{m}', compact_json(metadata),
             nim_sequence(orders),
             f'{inputs.nrows()}:{inputs.ncols()}:'+nim_sequence(inputs.list())]
    for name in spec['variables']:
        if name in operators:
            parts.append(name+':'+nim_sequence(operators[name].list()))
    return hashlib.sha1(''.join(parts).encode()).hexdigest().upper()

def load_record(directory, binding, relation, metadata, count, modulus):
    """
    Read the recorded choices for a single presented relation.
    Check their identifier, source, input count and working modulus.
    These checks do not replace the defining division and terminal equations.
    """
    suffix = hashlib.sha1(relation['name'].encode()).hexdigest().upper()
    path = directory / f'{binding}_{suffix}.json.gz'
    if not path.is_file():
        path = SUPPLEMENTARY_VERIFICATION_DIRECTORY / path.name
    with gzip.open(path,'rt') as stream:
        record = json.load(stream)
    assert record['schema'] in ('hecke.compact-relation-witness.v1','hecke.compact-relation-witness.v2')
    assert record['binding'] == binding and record['relation'] == relation['name']
    assert record['source'] == metadata and record['input_count'] == count
    assert record['working_modulus'] == modulus
    assert isinstance(record['independent'], bool)
    return record

def defining_equations(spec, relation, independent=False):
    """
    Expand divisions, polynomial presentations and literal compositions.
    Include the initial selector as a presented relation, not an ordinary
    matrix. Apply rightmost factors first and retain both selector copies.
    The recorded flag chooses shared or independent intermediate elements.
    Append the equation expressing membership in p^b M.
    """
    variables = spec['variables']
    definitions = {item['variable']:item for item in spec['divisions']}
    aliases = {item['variable']:item for item in spec.get('presentations',[])}
    steps, cache = [], {}
    def polynomial(terms, input_node):
        """Compile expanded monomials in archive order, combining equal outputs."""
        collected = {}
        for coefficient,powers in terms:
            key = tuple(powers)
            collected[key] = (collected.get(key,0)+ZZ(coefficient)) % spec['coefficient_modulus']
        outputs = {}
        for powers,coefficient in collected.items():
            if not coefficient:
                continue
            node = input_node
            for name,power in reversed(list(zip(variables,powers))):
                for _ in range(power):
                    node = apply_relation(name,node)
            outputs[node] = (outputs.get(node,0)+coefficient) % spec['coefficient_modulus']
        return [(node,c) for node,c in outputs.items() if c]
    def apply_relation(name, input_node):
        """Append one application, expanding compositions without adding a false division."""
        key = (name,input_node)
        ordinary = name in spec['hecke_operators']
        share = ordinary or not independent
        if share and key in cache:
            return cache[key]
        if ordinary:
            step = (name,input_node,0,None)
        elif name in aliases and 'factors' in aliases[name]:
            node = input_node
            for factor in reversed(aliases[name]['factors']):
                node = apply_relation(factor,node)
            if share:
                cache[key] = node
            return node
        else:
            definition = definitions.get(name,aliases.get(name))
            divisor = p^definition['power'] if name in definitions else 1
            terms = polynomial(definition.get('numerator',definition.get('polynomial')),input_node)
            step = (name,input_node,divisor,terms)
        steps.append(step)
        node = len(steps)
        if share:
            cache[key] = node
        return node
    initial = apply_relation(relation['input_presentation'],0) if 'input_presentation' in relation else 0
    terms = polynomial(relation['polynomial'],initial)
    steps.append((None,0,p^relation['terminal_power'],terms))
    return steps

def restore_outputs(spec, record, operators, orders, ring):
    """
    Decode reusable division outputs when the recorded file provides them.
    Verify each numerator equation and cyclic well-definedness. Compute
    polynomial aliases and ordered products from these checked matrices.
    They prescribe intermediate choices; the original equations are still
    replayed at every occurrence. Return no matrices for elementwise choices.
    """
    if 'recipe' not in record:
        return {}
    recipe = record['recipe']
    assert recipe['schema'] == 'hecke.global-division-recipe.v1'
    saved = iter(recipe['outputs'])
    matrices = dict(operators)
    definitions = {item['variable']:item for item in spec['divisions']}
    aliases = {item['variable']:item for item in spec.get('presentations',[])}
    rank = len(orders)
    for index,name in enumerate(spec['variables']):
        if name in operators:
            continue
        if name in aliases:
            alias = aliases[name]
            if 'factors' in alias:
                action = identity_matrix(ring,rank)
                for factor in reversed(alias['factors']):
                    action *= matrices[factor]
            else:
                action = polynomial_matrix(alias['polynomial'],spec['variables'],matrices,ring,rank)
            matrices[name] = action
            continue
        definition = definitions[name]
        numerator = polynomial_matrix(definition['numerator'],spec['variables'],matrices,ring,rank)
        item = next(saved)
        assert item['variable'] == index and len(item['entries']) == rank*rank
        assert all(0 <= entry < orders[k%rank] for k,entry in enumerate(item['entries']))
        chosen = matrix(ring,rank,rank,item['entries'])
        assert mixed_zero(p^definition['power']*chosen-numerator,orders)
        assert all(orders[i]*ZZ(chosen[i,j]) % orders[j] == 0
                   for i in range(rank) for j in range(rank))
        matrices[name] = chosen
    assert next(saved,None) is None
    return matrices

def replay_equations(spec, relation, record, inputs, operators, orders, ring):
    """
    Recover and check the intermediate elements on the prescribed input rows.
    Apply an ordinary input selector first. Reconstruct each recorded kernel
    correction or reusable division output, checking every division equation
    and the final membership in 5^b M. No equation is solved by search.
    """
    steps = defining_equations(spec,relation,record['independent'])
    reusable = restore_outputs(spec,record,operators,orders,ring)
    corrections = {}
    previous = None
    for node,row,col,coefficient in record['choices']:
        key = (node,row,col)
        assert previous is None or previous < key
        previous = key
        assert 1 <= node < len(steps) and 0 <= row < inputs.nrows() and 0 <= col < len(orders)
        divisor = steps[node-1][2]
        assert divisor > 1 and 0 < coefficient < gcd(divisor,orders[col])
        corrections[key] = coefficient
    assert not (reusable and corrections)
    initial = inputs
    if 'input_polynomial' in relation:
        initial = inputs*polynomial_matrix(relation['input_polynomial'],spec['variables'],
                                           operators,ring,len(orders))
    uses = [0]*(len(steps)+1)
    for name,input_node,divisor,terms in steps:
        dependencies = [input_node] if divisor == 0 else [node for node,c in terms]
        if reusable and divisor > 1 and name is not None:
            dependencies = dependencies+[input_node]
        for node in dependencies:
            uses[node] += 1
    for start in range(0,initial.nrows(),GENERATOR_BATCH_SIZE):
        count = min(GENERATOR_BATCH_SIZE,initial.nrows()-start)
        values = {0:initial.matrix_from_rows(range(start,start+count))}
        remaining = list(uses)
        for number,(name,input_node,divisor,terms) in enumerate(steps,start=1):
            if divisor == 0:
                output = values[input_node]*operators[name]
                dependencies = [input_node]
            else:
                rhs = zero_matrix(ring,count,len(orders))
                for node,c in terms:
                    rhs += c*values[node]
                output = canonical_preimage(rhs,divisor,orders)
                if reusable and divisor > 1 and name is not None:
                    output = values[input_node]*reusable[name]
                else:
                    for i in range(count):
                        for j,order in enumerate(orders):
                            coefficient = corrections.get((number,start+i,j),0)
                            output[i,j] += (order//gcd(divisor,order))*coefficient
                assert mixed_zero(divisor*output-rhs,orders), (relation['name'],number)
                dependencies = [node for node,c in terms]
                if reusable and divisor > 1 and name is not None:
                    dependencies = dependencies+[input_node]
            values[number] = output
            for node in dependencies:
                remaining[node] -= 1
                if remaining[node] == 0:
                    del values[node]
            if remaining[number] == 0:
                del values[number]

verified_cases = {}

def replay_mod125_case(group_name, d, q):
    """
    Verify the whole finite source using its recorded recursive presentation.
    First verify each required lower source, then the transfer homomorphisms
    and Hecke compatibility. Replay the selected complement and check that
    it completes the inherited rows to generators. Cache successful results
    only for this notebook session; missing or invalid data stop verification.
    """
    key = (group_name,int(d),int(q))
    if key in verified_cases:
        return verified_cases[key]
    group = groups[group_name]
    assert d in group['degrees'] and q in range(4)
    ring,m = group['ring'],group['exponent']
    orders,operators,maps,recursive = load_source(group_name,int(d),int(q))
    rank = len(orders)
    spec = group['specifications'][(d+50*q)%100]
    inherited = zero_matrix(ring,0,rank)
    if recursive:
        for label,shift,lower_q in [('a',5^m*4,q),('b',5^(m-1)*6,(q+1)%4)]:
            lower_d = d-shift
            if lower_d < 0:
                continue
            lower_spec = group['specifications'][(lower_d+50*lower_q)%100]
            assert lower_spec == spec
            replay_mod125_case(group_name,lower_d,lower_q)
            lower_orders,lower_operators,_,_ = load_source(group_name,int(lower_d),int(lower_q))
            transfer = maps['transfer_'+label]
            assert transfer.dimensions() == (len(lower_orders),rank)
            assert all(lower_orders[i]*ZZ(transfer[i,j]) % orders[j] == 0
                       for i in range(len(lower_orders)) for j in range(rank))
            for name in operators:
                assert mixed_zero(lower_operators[name]*transfer-transfer*operators[name],orders)
            inherited = inherited.stack(transfer)
        inputs = select_complement(inherited,maps['complement_images'],rank,ring)
    else:
        inputs = identity_matrix(ring,rank)
    metadata = {'degree':int(d),'orientation':int(q),'sign':int((-1)^q),
                'source_scope':'manin','construction':'recursive',
                'ideal':None,'ideal_variable_hecke_indices':[int(2),int(19)]}
    binding = packet_binding(spec,metadata,orders,operators,m,inputs)
    for relation in spec['relations']:
        record = load_record(VERIFICATION_DIRECTORY/'packets',binding,
                             relation,metadata,inputs.nrows(),5^m)
        replay_equations(spec,relation,record,inputs,operators,orders,ring)
    result = {'group':group_name,'degree':d,'orientation':q,'sign':(-1)^q,
              'rank':rank,'complement_inputs':inputs.nrows(),'recursive':recursive,
              'packets_loaded':len(spec['relations']),'passed':True}
    verified_cases[key] = result
    return result


## 3. Replay in ascending degree order

Check the lower untwisted plus cases first, then all four orientations of the induction base. Required lower orientations are also checked recursively. For a short trial, replace the case list with a few pairs, for example `[(26,0)]`.

A failed equation or missing file stops the loop. A successful finite loop verifies the requested source conditions, not the separate all-weight propagation or strong-realization argument.


In [14]:
cases = (
    [(d,0) for d in range(0,750,2)] +
    [(d,q) for d in range(750,3250,2) for q in range(4)]
)
assert len(cases) == 5375
results = []
for d,q in cases:
    print(f'Checking d={d}, q={q} ...',flush=True)
    test = replay_mod125_case('mod125',d,q)
    results.append(test)
    print(test,flush=True)
print('Cases replayed:',len(results))
print('ALL REQUESTED MODULO-125 FINITE-SOURCE CONDITIONS VERIFIED IN SAGE')


Checking d=0, q=0 ...
{'group': 'mod125', 'degree': 0, 'orientation': 0, 'sign': 1, 'rank': 0, 'complement_inputs': 0, 'recursive': False, 'packets_loaded': 8, 'passed': True}
Checking d=2, q=0 ...
{'group': 'mod125', 'degree': 2, 'orientation': 0, 'sign': 1, 'rank': 1, 'complement_inputs': 1, 'recursive': False, 'packets_loaded': 8, 'passed': True}
Checking d=4, q=0 ...
{'group': 'mod125', 'degree': 4, 'orientation': 0, 'sign': 1, 'rank': 1, 'complement_inputs': 1, 'recursive': False, 'packets_loaded': 8, 'passed': True}
Checking d=6, q=0 ...
{'group': 'mod125', 'degree': 6, 'orientation': 0, 'sign': 1, 'rank': 2, 'complement_inputs': 2, 'recursive': False, 'packets_loaded': 8, 'passed': True}
Checking d=8, q=0 ...
{'group': 'mod125', 'degree': 8, 'orientation': 0, 'sign': 1, 'rank': 1, 'complement_inputs': 1, 'recursive': False, 'packets_loaded': 8, 'passed': True}
Checking d=10, q=0 ...
{'group': 'mod125', 'degree': 10, 'orientation': 0, 'sign': 1, 'rank': 2, 'complement_inputs': 2,

## Optional: replay the same recorded data with Nim

This is an alternative to the Sage loops. Run the imports and relation definitions first. Set `RUN_NIM_REPLAY=True` to enable it and adjust `NIM_WORKERS` (default 4).

Each case checks the recursive lower sources as well as its complement, using the supplementary lower-minus files when needed. Fresh processes do not share the Sage cache and may repeat lower-degree work. Only saved intermediate elements are replayed; no new choices are searched for. After a failed batch, no further batch is started. The explicit Sage replay is intended for readability; the Nim option is preferable for speed on the full range.


In [15]:
import os
import subprocess
from concurrent.futures import ProcessPoolExecutor, as_completed
from multiprocessing import get_context

RUN_NIM_REPLAY = True
NIM_WORKERS = 4
VERIFIER = ROOT / 'nim/.verify-hecke-relations-build/verify_hecke_relations'

def replay_nim_case(case):
    """
    Replay one case and its recursive dependencies with the native verifier.
    Load archived sources and recorded intermediate elements in strict replay
    mode. Return the report; missing data or failed equations raise an error.
    """
    name,d,q = case
    group = groups[name]
    request = {
        'compute': {
            'prime':int(5),'exponent':int(group['exponent']),
            'degree':int(d),'orientation':int(q),
            'recursive':True,'recursive_verification':True,
            'archive_directory':str(group['sources']),
            'supplementary_archive_directories':[str(SUPPLEMENTARY_SOURCE_DIRECTORY)],
        },
        'relations':group['specifications'][(d+50*q)%100],
        'witness_directory':str(VERIFICATION_DIRECTORY/'packets'),
        'witness_mode':'replay',
        'witness_read_directories':[str(SUPPLEMENTARY_VERIFICATION_DIRECTORY)],
    }
    if not VERIFIER.is_file():
        raise FileNotFoundError(VERIFIER)
    environment = os.environ.copy()
    environment['LD_LIBRARY_PATH'] = (str(Path(sys.prefix)/'lib')+os.pathsep+
                                      environment.get('LD_LIBRARY_PATH',''))
    process = subprocess.run([str(VERIFIER),'-'],input=json.dumps(request),
                             text=True,capture_output=True,env=environment)
    if not process.stdout.strip():
        raise RuntimeError(process.stderr)
    report = json.loads(process.stdout)
    if process.returncode or not report.get('passed',False):
        raise RuntimeError(f'{case}: {report}')
    assert report['verification_scope'] == 'whole_source'
    assert report['recursive_verification'] is True
    assert report['archived_actions_reused'] is True
    assert len(report['relations']) == len(request['relations']['relations'])
    for result in report['relations']:
        assert result['passed'] is True and result['witness_file_reused'] is True
    for item in report['recursive_trace']:
        assert item['state'] == 'passed'
    return {'group':name,'degree':d,'orientation':q,'passed':True}

def run_parallel_nim_replay(cases, workers=4):
    """
    Replay cases in bounded parallel batches with an adjustable worker count.
    A fresh pool for each batch releases worker resources afterwards.
    Stop after a failed batch without scheduling new cases; return successful
    reports only if every requested case has passed.
    """
    if workers != int(workers) or workers < 1:
        raise ValueError('workers must be positive')
    workers = int(workers)
    results = []
    for start in range(0,len(cases),workers):
        batch = cases[start:start+workers]
        failures = []
        with ProcessPoolExecutor(max_workers=len(batch),
                                 mp_context=get_context('fork')) as executor:
            futures = {executor.submit(replay_nim_case,case):case for case in batch}
            for future in as_completed(futures):
                try:
                    result = future.result()
                    results.append(result)
                    print(result,flush=True)
                except Exception as error:
                    failures.append((futures[future],str(error)))
        if failures:
            raise RuntimeError(f'Batch failed; no further cases scheduled: {failures}')
    return results

if RUN_NIM_REPLAY:
    nim_cases = [
        (name,int(d),int(q))
        for name,group in groups.items()
        for d in group['degrees']
        for q in (range(4) if d >= group['threshold'] else (0,))
    ]
    nim_results = run_parallel_nim_replay(nim_cases,NIM_WORKERS)
    print('ALL REQUESTED FINITE-SOURCE CONDITIONS VERIFIED BY NIM REPLAY')
else:
    print('Optional Nim replay is disabled. Set RUN_NIM_REPLAY = True to run it.')


{'group': 'mod125', 'degree': 0, 'orientation': 0, 'passed': True}
{'group': 'mod125', 'degree': 6, 'orientation': 0, 'passed': True}
{'group': 'mod125', 'degree': 4, 'orientation': 0, 'passed': True}
{'group': 'mod125', 'degree': 2, 'orientation': 0, 'passed': True}
{'group': 'mod125', 'degree': 8, 'orientation': 0, 'passed': True}
{'group': 'mod125', 'degree': 10, 'orientation': 0, 'passed': True}
{'group': 'mod125', 'degree': 14, 'orientation': 0, 'passed': True}
{'group': 'mod125', 'degree': 12, 'orientation': 0, 'passed': True}
{'group': 'mod125', 'degree': 22, 'orientation': 0, 'passed': True}
{'group': 'mod125', 'degree': 20, 'orientation': 0, 'passed': True}
{'group': 'mod125', 'degree': 16, 'orientation': 0, 'passed': True}
{'group': 'mod125', 'degree': 18, 'orientation': 0, 'passed': True}
{'group': 'mod125', 'degree': 24, 'orientation': 0, 'passed': True}
{'group': 'mod125', 'degree': 28, 'orientation': 0, 'passed': True}
{'group': 'mod125', 'degree': 26, 'orientation': 0, '

## Strong realization of the permitted signatures

Check that every permitted signature has a saved strong representative.
Use the same parameter rows as in the displayed relations. For a
degree residue $r=k-2\bmod100$, put $\lambda_r=v_r\bmod5$. The
permitted $t\bmod25$ are $v_r$ and all lifts of
$\lambda_r\pm2\bmod5$. For each such $t$, solve
$$
a^2=h_{r,0}+5h_{r,1}t+25h_{r,2}t^2\pmod{125}.
$$
This gives 22 pairs $(a_2,a_{19})=(a,5t)$ in each even **weight**
residue $k\bmod100$, hence 1100 signatures in total.

Load their strong representatives from `strong_signatures/p5_m3/`
and check that they occur by weight 598. The loader checks file
digests and matches each summary
entry to its weight, eigenform orbit and prime above 5. The digests
check file consistency; they are not arithmetic proofs. This cell
does not recompute eigenforms or repeat their local KRW reductions.

Run the imports and relation definitions first. These cells use the
saved scan and do not start a new computation or rerun the source
replay. The comparison supplies the finite realization step. The
all-weight conclusion also requires the complete source checks,
propagation and completed-Hecke generation from the manuscript.


In [6]:
from p5_mod125 import mod125_possible_signatures, load_mod125_strong_representatives

STRONG_SIGNATURE_DIRECTORY_MOD125 = ROOT / 'strong_signatures/p5_m3'
STRONG_WEIGHT_BOUND_MOD125 = 598
STRONG_WEIGHT_PERIOD_MOD125 = 100

# Use the same degree-indexed parameter rows as the source relations.
# The returned dictionary is indexed by weight residue r+2.
strong_mod125_expected = mod125_possible_signatures(parameters)
assert set(strong_mod125_expected) == set(range(0, 100, 2))
assert all(len(pairs) == 22 for pairs in strong_mod125_expected.values())
assert sum(map(len, strong_mod125_expected.values())) == 1100

# Check the residue-field polynomial identity symbolically, not only
# on the five values in F_5. This is the identity used in the proof.
strong_polynomial_ring = PolynomialRing(GF(5), 'U')
strong_U = strong_polynomial_ring.gen()
for strong_scalar in GF(5):
    assert prod(strong_U^2 - strong_scalar*strong_U - t^2 + strong_scalar*t
                for t in GF(5)) == (strong_U^5 - strong_U)^2
print('Residue-field polynomial identity verified.')


Residue-field polynomial identity verified.


In [7]:
# Match each saved representative to its recorded local KRW reduction.
# Keep the working modulus 625 above unchanged: these signatures use 125.
strong_mod125_signatures = load_mod125_strong_representatives(
    STRONG_SIGNATURE_DIRECTORY_MOD125)
strong_mod125_realized = {}
for weight, a2, a19 in strong_mod125_signatures:
    assert 2 <= weight <= STRONG_WEIGHT_BOUND_MOD125 and weight % 2 == 0
    k_residue = weight % STRONG_WEIGHT_PERIOD_MOD125
    strong_mod125_realized.setdefault(k_residue, set()).add((ZZ(a2), ZZ(a19)))
print('Saved strong representatives:', len(strong_mod125_signatures))
print('Largest representative weight:', max(weight for weight, _, _ in strong_mod125_signatures))


Saved strong representatives: 1100
Largest representative weight: 598


In [8]:
print('k mod 100 | (a_2, a_19) mod 125')
print('-' * 100)
for k_residue in sorted(strong_mod125_expected):
    pairs = ', '.join(
        f'({a1}, {a2})'
        for a1, a2 in sorted(strong_mod125_realized.get(k_residue, set()))
    )
    print(f'{k_residue:3d} | {pairs}')

assert strong_mod125_realized == strong_mod125_expected
print('Every permitted modulo-125 signature has a strong representative.')


k mod 100 | (a_2, a_19) mod 125
----------------------------------------------------------------------------------------------------
  0 | (6, 40), (9, 120), (16, 70), (19, 90), (31, 115), (34, 45), (41, 20), (44, 15), (56, 65), (59, 95), (61, 80), (64, 80), (66, 95), (69, 65), (81, 15), (84, 20), (91, 45), (94, 115), (106, 90), (109, 70), (116, 120), (119, 40)
  2 | (3, 20), (7, 55), (12, 10), (13, 110), (18, 30), (32, 80), (37, 35), (38, 85), (43, 5), (57, 105), (62, 60), (63, 60), (68, 105), (82, 5), (87, 85), (88, 35), (93, 80), (107, 30), (112, 110), (113, 10), (118, 55), (122, 20)
  4 | (1, 90), (4, 100), (9, 110), (16, 60), (24, 15), (26, 40), (34, 35), (41, 10), (49, 65), (51, 115), (59, 85), (66, 85), (74, 115), (76, 65), (84, 10), (91, 35), (99, 40), (101, 15), (109, 60), (116, 110), (121, 100), (124, 90)
  6 | (2, 60), (8, 0), (12, 120), (13, 95), (17, 25), (33, 100), (37, 20), (38, 70), (42, 50), (58, 75), (62, 45), (63, 45), (67, 75), (83, 50), (87, 70), (88, 20), (92, 100